In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('phone_raw.csv')
df.head()

,source_site,scraped_at,product_url,image_url,brand,model_name,variant_id,product_id,price_egp,original_price_egp,in_stock,is_best_seller,is_new_arrival,seller_name,category_l3
0,btech,2026-07-23T03:08:05.139804,https://btech.com/en/p/82f1a686-4373-43d5-b79c...,4/6/5/f/465f167c9f4e964a29f028dd5247bae5f1273d...,Infinix,"Infinix Smart 20 , 64 GB , 4 GB , 4G , Dual-SI...",82f1a686-4373-43d5-b79c-19c02f8408ff,a6844b2a-ae27-4a09-8585-ce96917db568,6330.0,6330.0,True,True,False,Elhashim Store,Smart Phones
1,btech,2026-07-23T03:08:05.139848,https://btech.com/en/p/760705ec-59eb-4196-a844...,f/a/2/f/fa2f1ca443cbc3f983f400ef43cc035773dd4c...,Infinix,"Infinix Smart 20 , 128 GB , 4 GB , 4G , Dual-S...",760705ec-59eb-4196-a844-687dfb44e8c6,a6844b2a-ae27-4a09-8585-ce96917db568,6835.0,6835.0,True,True,False,Elhashim Store,Smart Phones
2,btech,2026-07-23T03:08:05.139867,https://btech.com/en/p/95ea8720-6e24-436b-bc06...,b/c/2/f/bc2f1f5328bae6e875c5faa2e4ca991d3d707a...,Samsung,"Samsung A07 , 64 GB , 4 GB , 4G LTE , Dual-SIM...",95ea8720-6e24-436b-bc06-a4cc2f09f5aa,c9a4d732-be6e-4ed4-bb9b-772d53e488f2,6798.0,6798.0,True,True,False,Remoz,Smart Phones
3,btech,2026-07-23T03:08:05.139882,https://btech.com/en/p/ba7338f2-0084-4d9b-b80b...,9/e/f/d/9efdd1545183b914e0f1c33c54dddfe491a139...,Samsung,"Samsung A07 , 128 GB , 6 GB , 4G LTE , Dual-SI...",ba7338f2-0084-4d9b-b80b-3664df6ac7dc,c9a4d732-be6e-4ed4-bb9b-772d53e488f2,8888.0,8888.0,True,True,False,Ehab Group,Smart Phones
4,btech,2026-07-23T03:08:05.139897,https://btech.com/en/p/012251d6-c872-47db-be03...,a/9/8/9/a989cb00c239c01e0a04be611bcc20fa32ab30...,Infinix,"Infinix Smart 20 , 128 GB , 4 GB , 4G , Dual-S...",012251d6-c872-47db-be03-c4e12687f205,a6844b2a-ae27-4a09-8585-ce96917db568,6830.0,6930.0,True,True,False,B.TECH,Smart Phones


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 663 entries, 0 to 662
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   source_site         663 non-null    str    
 1   scraped_at          663 non-null    str    
 2   product_url         663 non-null    str    
 3   image_url           663 non-null    str    
 4   brand               663 non-null    str    
 5   model_name          663 non-null    str    
 6   variant_id          663 non-null    str    
 7   product_id          663 non-null    str    
 8   price_egp           663 non-null    float64
 9   original_price_egp  663 non-null    float64
 10  in_stock            663 non-null    bool   
 11  is_best_seller      663 non-null    bool   
 12  is_new_arrival      663 non-null    bool   
 13  seller_name         663 non-null    str    
 14  category_l3         663 non-null    str    
dtypes: bool(3), float64(2), str(10)
memory usage: 64.2 KB


In [3]:
df.describe()

,price_egp,original_price_egp
count,663.000000,663.000000
mean,28754.284976,28858.888295
std,33152.961020,33312.542528
min,475.000000,475.000000
25%,8812.000000,8829.000000
50%,15550.000000,15583.000000
75%,29943.500000,30110.500000
max,189999.000000,189999.000000


In [4]:

import re
import pandas as pd


def parse_model_name(raw: str) -> dict:
    result = {"model": None, "storage_gb": None, "ram_gb": None,
              "network": None, "sim_config": None, "color": None}

    if " - " in raw:
        main_part, color = raw.rsplit(" - ", 1)
        result["color"] = color.strip()
    else:
        main_part = raw

    parts = [p.strip() for p in main_part.split(",")]
    result["model"] = parts[0]

    for p in parts[1:]:
        if p.endswith("GB"):
            number_str = p.replace("GB", "").strip()
            if number_str.isdigit():
                if result["storage_gb"] is None:
                    result["storage_gb"] = int(number_str)
                else:
                    result["ram_gb"] = int(number_str)
        elif p.endswith("Terabyte"):
            number_str = p.replace("Terabyte", "").strip()
            if number_str.isdigit():
                result["storage_gb"] = int(number_str) * 1024
        elif "SIM" in p:
            result["sim_config"] = p
        elif p in ("5G", "4G", "4G LTE", "3G", "LTE"):
            result["network"] = p

    return result


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    parsed = df["model_name"].apply(parse_model_name).apply(pd.Series)
    out = pd.concat([df, parsed], axis=1)
    # Feature phones (basic Nokia/HMD/Itel handsets) genuinely have no
    # storage/RAM specs to scrape — flag rather than treat as parse failures.
    out["is_feature_phone"] = out["storage_gb"].isna() & out["ram_gb"].isna()
    return out


if __name__ == "__main__":
    df = pd.read_csv("phone_raw.csv")
    cleaned = clean_dataframe(df)

    # Only storage/RAM/model are "core" — network and sim_config are
    # legitimately absent on some real listings, not parse bugs.
    core_cols = ["model", "storage_gb", "ram_gb"]
    smartphones = cleaned[~cleaned["is_feature_phone"]]
    failures = smartphones[smartphones[core_cols].isna().any(axis=1)]

    print(f"Total rows: {len(cleaned)}")
    print(f"Feature phones (no storage/RAM specs, flagged not failed): {cleaned['is_feature_phone'].sum()}")
    print(f"Smartphones with a genuine parse failure: {len(failures)}")
    if len(failures) > 0:
        print("\nSample failures:")
        print(failures[["model_name"] + core_cols].head(10).to_string())

    print("\nSample successful parses:")
    print(cleaned[["model_name", "model", "storage_gb", "ram_gb", "network", "sim_config", "color"]].head(8).to_string())

    cleaned.to_csv("phone_cleaned.csv", index=False)
    print(f"\nSaved phone_cleaned.csv ({len(cleaned)} rows)")

Total rows: 663
Feature phones (no storage/RAM specs, flagged not failed): 44
Smartphones with a genuine parse failure: 0

Sample successful parses:
                                                              model_name               model  storage_gb  ram_gb network sim_config             color
0         Infinix Smart 20 , 64 GB , 4 GB , 4G , Dual-SIM - Shadow Black    Infinix Smart 20        64.0     4.0      4G   Dual-SIM      Shadow Black
1    Infinix Smart 20 , 128 GB , 4 GB , 4G , Dual-SIM - Polaris Titanium    Infinix Smart 20       128.0     4.0      4G   Dual-SIM  Polaris Titanium
2                 Samsung A07 , 64 GB , 4 GB , 4G LTE , Dual-SIM - Black         Samsung A07        64.0     4.0  4G LTE   Dual-SIM             Black
3                Samsung A07 , 128 GB , 6 GB , 4G LTE , Dual-SIM - Black         Samsung A07       128.0     6.0  4G LTE   Dual-SIM             Black
4        Infinix Smart 20 , 128 GB , 4 GB , 4G , Dual-SIM - Shadow Black    Infinix Smart 20       12

In [5]:
cleaned.head()

,source_site,scraped_at,product_url,image_url,brand,model_name,variant_id,product_id,price_egp,original_price_egp,...,is_new_arrival,seller_name,category_l3,model,storage_gb,ram_gb,network,sim_config,color,is_feature_phone
0,btech,2026-07-23T03:08:05.139804,https://btech.com/en/p/82f1a686-4373-43d5-b79c...,4/6/5/f/465f167c9f4e964a29f028dd5247bae5f1273d...,Infinix,"Infinix Smart 20 , 64 GB , 4 GB , 4G , Dual-SI...",82f1a686-4373-43d5-b79c-19c02f8408ff,a6844b2a-ae27-4a09-8585-ce96917db568,6330.0,6330.0,...,False,Elhashim Store,Smart Phones,Infinix Smart 20,64.0,4.0,4G,Dual-SIM,Shadow Black,False
1,btech,2026-07-23T03:08:05.139848,https://btech.com/en/p/760705ec-59eb-4196-a844...,f/a/2/f/fa2f1ca443cbc3f983f400ef43cc035773dd4c...,Infinix,"Infinix Smart 20 , 128 GB , 4 GB , 4G , Dual-S...",760705ec-59eb-4196-a844-687dfb44e8c6,a6844b2a-ae27-4a09-8585-ce96917db568,6835.0,6835.0,...,False,Elhashim Store,Smart Phones,Infinix Smart 20,128.0,4.0,4G,Dual-SIM,Polaris Titanium,False
2,btech,2026-07-23T03:08:05.139867,https://btech.com/en/p/95ea8720-6e24-436b-bc06...,b/c/2/f/bc2f1f5328bae6e875c5faa2e4ca991d3d707a...,Samsung,"Samsung A07 , 64 GB , 4 GB , 4G LTE , Dual-SIM...",95ea8720-6e24-436b-bc06-a4cc2f09f5aa,c9a4d732-be6e-4ed4-bb9b-772d53e488f2,6798.0,6798.0,...,False,Remoz,Smart Phones,Samsung A07,64.0,4.0,4G LTE,Dual-SIM,Black,False
3,btech,2026-07-23T03:08:05.139882,https://btech.com/en/p/ba7338f2-0084-4d9b-b80b...,9/e/f/d/9efdd1545183b914e0f1c33c54dddfe491a139...,Samsung,"Samsung A07 , 128 GB , 6 GB , 4G LTE , Dual-SI...",ba7338f2-0084-4d9b-b80b-3664df6ac7dc,c9a4d732-be6e-4ed4-bb9b-772d53e488f2,8888.0,8888.0,...,False,Ehab Group,Smart Phones,Samsung A07,128.0,6.0,4G LTE,Dual-SIM,Black,False
4,btech,2026-07-23T03:08:05.139897,https://btech.com/en/p/012251d6-c872-47db-be03...,a/9/8/9/a989cb00c239c01e0a04be611bcc20fa32ab30...,Infinix,"Infinix Smart 20 , 128 GB , 4 GB , 4G , Dual-S...",012251d6-c872-47db-be03-c4e12687f205,a6844b2a-ae27-4a09-8585-ce96917db568,6830.0,6930.0,...,False,B.TECH,Smart Phones,Infinix Smart 20,128.0,4.0,4G,Dual-SIM,Shadow Black,False


In [6]:
cleaned = cleaned.drop(columns=["model_name", "scraped_at", "is_feature_phone"])

In [7]:
cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 663 entries, 0 to 662
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   source_site         663 non-null    str    
 1   product_url         663 non-null    str    
 2   image_url           663 non-null    str    
 3   brand               663 non-null    str    
 4   variant_id          663 non-null    str    
 5   product_id          663 non-null    str    
 6   price_egp           663 non-null    float64
 7   original_price_egp  663 non-null    float64
 8   in_stock            663 non-null    bool   
 9   is_best_seller      663 non-null    bool   
 10  is_new_arrival      663 non-null    bool   
 11  seller_name         663 non-null    str    
 12  category_l3         663 non-null    str    
 13  model               663 non-null    str    
 14  storage_gb          619 non-null    float64
 15  ram_gb              619 non-null    float64
 16  network            

In [8]:
cleaned.to_csv("phone_cleaned.csv", index=False)

In [9]:
df = pd.read_csv("phone_cleaned.csv")
df.describe()

,price_egp,original_price_egp,storage_gb,ram_gb
count,663.000000,663.000000,619.000000,619.000000
mean,28754.284976,28858.888295,250.702746,8.308562
std,33152.961020,33312.542528,196.479267,5.685681
min,475.000000,475.000000,8.000000,2.000000
25%,8812.000000,8829.000000,128.000000,6.000000
50%,15550.000000,15583.000000,256.000000,8.000000
75%,29943.500000,30110.500000,256.000000,12.000000
max,189999.000000,189999.000000,2048.000000,128.000000


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 663 entries, 0 to 662
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   source_site         663 non-null    str    
 1   product_url         663 non-null    str    
 2   image_url           663 non-null    str    
 3   brand               663 non-null    str    
 4   variant_id          663 non-null    str    
 5   product_id          663 non-null    str    
 6   price_egp           663 non-null    float64
 7   original_price_egp  663 non-null    float64
 8   in_stock            663 non-null    bool   
 9   is_best_seller      663 non-null    bool   
 10  is_new_arrival      663 non-null    bool   
 11  seller_name         663 non-null    str    
 12  category_l3         663 non-null    str    
 13  model               663 non-null    str    
 14  storage_gb          619 non-null    float64
 15  ram_gb              619 non-null    float64
 16  network            

In [11]:
df["ram_gb"].value_counts()

ram_gb
8.0      252
12.0     170
4.0       95
6.0       68
3.0       21
2.0        6
16.0       6
128.0      1
Name: count, dtype: int64

In [15]:
outliers = df[df["ram_gb"] > 12]   
print(f"Outliers (RAM > 12GB): {len(outliers)}")
for index, row in outliers.iterrows():
    print(f"Model: {row['variant_id']},{row['model']}, RAM: {row['ram_gb']}GB, Storage: {row['storage_gb']}GB, Network: {row['network']}, SIM Config: {row['sim_config']}, Color: {row['color']}")

Outliers (RAM > 12GB): 7
Model: 95f43733-72b9-4355-b2c8-bb08b34bf4c1,Samsung Galaxy S26 Ultra, RAM: 16.0GB, Storage: 1024.0GB, Network: 5G, SIM Config: Dual-SIM, Color: Cobalt Violet
Model: 773772b4-f6e0-40d4-8623-9ede49eeedfc,Honor Magic V5, RAM: 16.0GB, Storage: 512.0GB, Network: 5G, SIM Config: Dual-SIM, Color: Ivory White
Model: dc85d2cd-e0a5-4648-997d-42a21e72de7c,Samsung Galaxy A37, RAM: 128.0GB, Storage: 8.0GB, Network: 5G, SIM Config: Dual-SIM, Color: Awesome Green Grey
Model: c4340dd0-c6e7-434e-aed0-62f3861356bf,Samsung Galaxy S26 Ultra, RAM: 16.0GB, Storage: 1024.0GB, Network: 5G, SIM Config: Dual-SIM, Color: Black
Model: 3ef29c15-0242-48fd-ba85-02b152cd09c9,Oppo Find X9 Pro, RAM: 16.0GB, Storage: 512.0GB, Network: 5G, SIM Config: Dual-SIM, Color: Silk White
Model: d914f95b-7825-4d01-aeaa-70fe41021103,Oppo Find X9 Pro, RAM: 16.0GB, Storage: 512.0GB, Network: 5G, SIM Config: Dual-SIM, Color: Titanium Charcoal
Model: bc63ae41-a40c-4d78-ad98-b01f87d6f854,Honor Magic V5, RAM: 16.

In [16]:
mask = df["variant_id"] == "dc85d2cd-e0a5-4648-997d-42a21e72de7c"

df.loc[mask, ["ram_gb", "storage_gb"]] = df.loc[mask, ["storage_gb", "ram_gb"]].values

# verify
print(df[mask][["brand", "model", "ram_gb", "storage_gb"]])

       brand               model  ram_gb  storage_gb
354  Samsung  Samsung Galaxy A37     8.0       128.0


In [ ]:
outliers = df[df["ram_gb"] > df["storage_gb"].quantile(0.99)]
print(f"Outliers (RAM > storage): {len(outliers)}")
for index, row in outliers.iterrows():
    print(f"Model: {row['variant_id']},{row['model']}, RAM: {row['ram_gb']}GB, Storage: {row['storage_gb']}GB, Network: {row['network']}, SIM Config: {row['sim_config']}, Color: {row['color']}")

Outliers (RAM > storage): 0


In [21]:
df.to_csv("phone_cleaned.csv", index=False)

In [25]:
df["model"].nunique()

189